# Nettoyage et assemblage des données

Ici on applique les décisions prises dans les trois notebooks d'exploration (`bodacc.ipynb`, `sirene.ipynb`, `ratios.ipynb`). À chaque étape, on rappelle la décision et on regarde combien de lignes sont touchées.

Les fichiers propres sont enregistrés dans `data/processed/` :
- `sirene_clean.parquet` : une ligne par entreprise ;
- `ratios_clean.parquet` : une ligne par exercice comptable ;
- `bodacc_clean.parquet` : une ligne par annonce et par entreprise, avec le type d'événement ;
- `panel_exercices.parquet` : le tableau final, une ligne par exercice comptable, avec les infos SIRENE et l'étiquette « faillite dans les 36 mois ».

⚠️ **RAM** : on traite les sources **une par une**, on enregistre le résultat, puis on libère la mémoire (`del` + `free_memory()`) avant de passer à la suivante.

## 1. Préparation

In [1]:
import sys
sys.path.append("../scripts")

import numpy as np
import pandas as pd
from data_utils import (PROCESSED, load_sirene_ul, load_sirene_sieges, load_ratios,
                        naf_section, free_memory, mem)

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATE_FIN_DONNEES = pd.Timestamp("2026-10-31")  # les données s'arrêtent fin octobre 2026
print(mem())

process : 0.14 Go | disponible : 9.38 Go


## 2. SIRENE

Les décisions (voir le récapitulatif de `sirene.ipynb`) :

| # | Décision |
|---|---|
| 1 | On garde **toutes** les entreprises. |
| 2 | On garde les entreprises **fermées** et **purgées**. |
| 3 | On garde les entreprises en **diffusion partielle**. Le code postal masqué (`[ND]`) passe en valeur manquante. |
| 4 | La taille ne vient **pas** de SIRENE (effectifs `NN` à 94 %) : on ne garde pas les effectifs. |
| 5 | Secteur : code **NAFRev2**, avec la **section** (lettre) et la **division** (2 chiffres). Les codes dans une ancienne nomenclature passent en valeur manquante. Le code d'attente `00.00Z` (pas encore de vrai code d'activité) aussi. |
| 8 | Dates bizarres (avant 1900 ou après 2026) : on **répare** avec la date du siège quand elle est correcte, sinon **valeur manquante**, et on garde la ligne. |

In [2]:
ul = load_sirene_ul(columns=[
    "siren", "statutDiffusionUniteLegale", "unitePurgeeUniteLegale", "dateCreationUniteLegale", "dateDebut",
    "etatAdministratifUniteLegale", "categorieJuridiqueUniteLegale",
    "activitePrincipaleUniteLegale", "nomenclatureActivitePrincipaleUniteLegale",
])
sieges = load_sirene_sieges(columns=["siren", "codeCommuneEtablissement", "codePostalEtablissement",
                                     "codePaysEtrangerEtablissement", "dateCreationEtablissement", "dateDebut"])
sieges = sieges.rename(columns={"dateDebut": "dateDebutSiege"})
print(ul.shape, sieges.shape, "|", mem())

(30148401, 9) (30126745, 6) | process : 1.75 Go | disponible : 7.40 Go


### 2.1 Les dates (décision 8)

In [3]:
def en_date(s):
    """Texte 'AAAA-MM-JJ' -> date. Vide ou illisible -> valeur manquante."""
    return pd.to_datetime(s.astype("string"), format="%Y-%m-%d", errors="coerce")

def correcte(d):
    return d.dt.year.between(1900, 2026)

# On aligne les dates du siège sur les entreprises
s = ul[["siren"]].merge(sieges[["siren", "dateCreationEtablissement", "dateDebutSiege"]], on="siren", how="left")
crea_siege, debut_siege = en_date(s["dateCreationEtablissement"]), en_date(s["dateDebutSiege"])
del s

lignes = {}
for col, siege in [("dateCreationUniteLegale", crea_siege), ("dateDebut", debut_siege)]:
    texte = ul[col].astype("string")
    annee = pd.to_numeric(texte.str[:4], errors="coerce")
    bizarre = ((annee < 1900) | (annee > 2026)).fillna(False).astype(bool)
    # pd.to_datetime refuse les années impossibles (ex. 9202) : elles deviennent des valeurs manquantes
    d = en_date(texte).where(~bizarre)
    reparable = bizarre & correcte(siege).fillna(False)
    d = d.where(~reparable, siege)
    lignes[col] = {"bizarres": int(bizarre.sum()), "réparées avec le siège": int(reparable.sum()),
                   "passées en valeur manquante": int((bizarre & ~reparable).sum())}
    ul[col] = d

del crea_siege, debut_siege
free_memory()
ul = ul.rename(columns={"dateCreationUniteLegale": "date_creation", "dateDebut": "date_debut"})
pd.DataFrame(lignes).T

,bizarres,réparées avec le siège,passées en valeur manquante
dateCreationUniteLegale,1129,21,1108
dateDebut,1587,128,1459


### Observations

- Sur 1 129 dates de création bizarres, seulement **21 sont réparées** avec la date du siège, et 1 108 passent en valeur manquante.
- Sur 1 587 `dateDebut` bizarres, **128 sont réparées** et 1 459 passent en valeur manquante.
- Comme vu dans `sirene.ipynb`, ça ne touche que 2 entreprises du BODACC 2019 et 32 entreprises qui ont des ratios.

### 2.2 Le secteur (décision 5)

In [4]:
rev2 = ul["nomenclatureActivitePrincipaleUniteLegale"] == "NAFRev2"
code_naf = ul["activitePrincipaleUniteLegale"].astype("string")
attente = code_naf == "00.00Z"  # code d'attente de l'Insee : pas encore de vrai code d'activité
print("codes d'attente 00.00Z :", attente.sum())
ul["naf"] = code_naf.where(rev2 & ~attente).astype("category")
del code_naf, attente
ul["naf_division"] = ul["naf"].astype("string").str[:2].astype("category")
ul["naf_section"] = naf_section(ul["naf"])
ul = ul.drop(columns=["activitePrincipaleUniteLegale", "nomenclatureActivitePrincipaleUniteLegale"])

print("code NAF gardé :", ul["naf"].notna().sum(), "| passé en valeur manquante :", ul["naf"].isna().sum())
ul["naf_section"].value_counts(dropna=False)

codes d'attente 00.00Z : 92925


code NAF gardé : 24021882 | passé en valeur manquante : 6126519


naf_section
<NA>    6126519
L       4464954
G       3189040
M       2594297
F       1878838
S       1777779
N       1417867
Q       1250749
A       1164111
R       1069885
H       1015429
I       1015376
J        826138
C        761959
P        735498
K        586508
D        160656
O         64662
E         43071
B          4319
U           703
T            43
Name: count, dtype: int64

### Observations

- On garde un code NAF pour **24 021 882 entreprises**.
- **6 126 519** passent en valeur manquante : 6 033 594 parce qu'elles sont dans une ancienne nomenclature, et 92 925 parce qu'elles ont le code d'attente `00.00Z`.
- Les sections les plus fréquentes sont l'immobilier (L : 4,5 M), le commerce (G : 3,2 M) et les activités spécialisées (M : 2,6 M).

### 2.3 Forme juridique, état et localisation du siège (décisions 1, 2, 3)

In [5]:
ul["cj_famille"] = ul["categorieJuridiqueUniteLegale"].astype("string").str[0].astype("category")
ul["active"] = ul["etatAdministratifUniteLegale"] == "A"
ul["purgee"] = ul["unitePurgeeUniteLegale"] == "true"
ul["diffusion_partielle"] = ul["statutDiffusionUniteLegale"] == "P"
ul = ul.drop(columns=["etatAdministratifUniteLegale", "unitePurgeeUniteLegale", "statutDiffusionUniteLegale"])

loc = sieges[["siren", "codeCommuneEtablissement", "codePostalEtablissement", "codePaysEtrangerEtablissement"]].copy()
del sieges
commune = loc["codeCommuneEtablissement"].astype("string").replace("", pd.NA)
loc["commune"] = commune.astype("category")
loc["departement"] = commune.str[:2].where(~commune.str.startswith("97"), commune.str[:3]).astype("category")
loc["code_postal"] = loc["codePostalEtablissement"].astype("string").replace({"": pd.NA, "[ND]": pd.NA}).astype("category")
loc["siege_etranger"] = loc["codePaysEtrangerEtablissement"] != ""
loc = loc[["siren", "commune", "departement", "code_postal", "siege_etranger"]]

ul = ul.merge(loc, on="siren", how="left")
del loc
free_memory()
ul = ul.rename(columns={"categorieJuridiqueUniteLegale": "categorie_juridique"})
print(ul.shape, f"{ul.memory_usage(deep=True).sum() / 1e9:.2f} Go |", mem())
ul.head()

(30148401, 15) 2.32 Go | process : 3.23 Go | disponible : 6.18 Go


,siren,date_creation,date_debut,categorie_juridique,naf,naf_division,naf_section,cj_famille,active,purgee,diffusion_partielle,commune,departement,code_postal,siege_etranger
0,325175,2000-09-26,2018-02-07,1000,32.12Z,32,C,1,True,False,False,13204,13,13004,False
1,1807254,1972-05-01,2014-12-31,1000,85.59A,85,P,1,False,False,False,02691,02,02100,False
2,5410220,1954-12-25,1988-03-31,1000,<NA>,<NA>,<NA>,1,False,True,False,80765,80,80210,False
3,5410345,NaT,1984-12-25,1000,<NA>,<NA>,<NA>,1,False,True,False,80001,80,80100,False
4,5410394,1954-12-25,1987-12-01,1000,<NA>,<NA>,<NA>,1,False,True,False,80025,80,80120,False


In [6]:
# Vérifications rapides
print("SIREN en double :", ul["siren"].duplicated().sum())
print("code postal manquant chez les diffusions partielles :", ul.loc[ul["diffusion_partielle"], "code_postal"].isna().mean().round(3))
ul.isna().mean().round(3)

SIREN en double : 0
code postal manquant chez les diffusions partielles : 1.0


siren                 0.00
date_creation         0.04
date_debut            0.01
categorie_juridique   0.00
naf                   0.20
naf_division          0.20
naf_section           0.20
cj_famille            0.00
active                0.00
purgee                0.00
diffusion_partielle   0.00
commune               0.01
departement           0.01
code_postal           0.13
siege_etranger        0.00
dtype: float64

### Observations

- Chaque entreprise apparaît **une seule fois**.
- Le code postal manque pour 13 % des entreprises. Ça comprend toutes les entreprises en diffusion partielle (100 %), comme prévu.
- Le code NAF manque pour 20 % des entreprises. Ce sont surtout des entreprises fermées (85 % d'entre elles).
- Le fichier `sirene_clean.parquet` contient **30 148 401 entreprises** et 15 colonnes. Il fait 487 Mo sur le disque et 2,3 Go en RAM.

In [7]:
ul.to_parquet(PROCESSED / "sirene_clean.parquet", index=False)
print("enregistré :", (PROCESSED / "sirene_clean.parquet").stat().st_size / 1e6, "Mo")
del ul
print(free_memory())

enregistré : 487.72801 Mo
process : 2.21 Go | disponible : 8.38 Go


## 3. Ratios financiers

Les décisions (voir le récapitulatif de `ratios.ipynb`) :

| # | Décision |
|---|---|
| 5 | On garde les exercices clôturés **de 2016 à aujourd'hui**. |
| 1 | On garde le bilan de la **société seule** (`C` ou `S`). Le bilan du **groupe** (`K`) seulement s'il n'y a pas de bilan de la société seule à la même date. |
| 2 | Si on a un bilan simplifié **et** un bilan complet à la même date, on garde le **complet** (`C`). |
| 3 | Comptes **publics** : les 0 restent à 0 €. Comptes **confidentiels** (partiellement confidentiel, RAPCAC) : les 0 passent en valeur manquante. Les lignes en « publication simplifiée » sont traitées comme publiques. |
| 4 | **Winsorisation** colonne par colonne au 1 % le plus bas et au 1 % le plus haut, calculée **après** les autres étapes, avec une colonne oui/non `<colonne>_ramene`. |

In [8]:
r = load_ratios()
avant = len(r)
print(r.shape, "|", mem())

(6813145, 23) | process : 3.35 Go | disponible : 7.06 Go


### 3.1 Les années (décision 5)

In [9]:
garder = r["date_cloture_exercice"].between("2016-01-01", DATE_FIN_DONNEES)
print("exercices enlevés :", (~garder).sum())
display(r.loc[~garder, "date_cloture_exercice"].dt.year.value_counts().sort_index())
r = r[garder]

exercices enlevés : 164398


date_cloture_exercice
1919         1
2002         1
2003         1
2004         2
2005         1
2006         2
2007         3
2008         1
2009         7
2010        26
2011        47
2012      2831
2013     12582
2014     30033
2015    118859
2029         1
Name: count, dtype: int64

### Observations

- On enlève **164 398 exercices**, presque tous clôturés en 2015 (118 859) ou en 2014 (30 033), plus le 1919 et le 2029.

### 3.2 Un seul bilan par entreprise et par date (décisions 1 et 2)

In [10]:
# Ordre de préférence : complet (C), puis simplifié (S), puis groupe (K)
priorite = r["type_bilan"].astype("string").map({"C": 0, "S": 1, "K": 2})
r = r.assign(_p=priorite).sort_values(["siren", "date_cloture_exercice", "_p"])
doublon = r.duplicated(["siren", "date_cloture_exercice"], keep="first")
print("lignes enlevées :")
display(r.loc[doublon, "type_bilan"].value_counts())
r = r[~doublon].drop(columns="_p")
print("bilans de groupe (K) gardés car seuls à leur date :", (r["type_bilan"] == "K").sum())
print("doublons restants :", r.duplicated(["siren", "date_cloture_exercice"]).sum())

lignes enlevées :


type_bilan
K    29447
S     2098
C        0
Name: count, dtype: int64

bilans de groupe (K) gardés car seuls à leur date : 3152


doublons restants : 0


### Observations

- On enlève **29 447 bilans de groupe** (`K`) qui avaient un bilan de la société seule à la même date, et **2 098 bilans simplifiés** (`S`) qui avaient un bilan complet à la même date.
- **3 152 bilans de groupe** sont gardés, parce que c'était le seul bilan de l'entreprise à cette date.
- Il ne reste plus aucun doublon entreprise + date.

### 3.3 Les zéros des comptes confidentiels (décision 3)

In [11]:
NUM = r.select_dtypes("number").columns.drop("siren").tolist()
confidentiel = r["confidentiality"].isin(["Partiellement confidentiel", "Partiellement confidentiel (RAPCAC)"])
zeros = (r[NUM] == 0) & confidentiel.to_numpy()[:, None]
display(zeros.sum().rename("zéros passés en valeur manquante"))
r[NUM] = r[NUM].mask(zeros)
del zeros
free_memory()

chiffre_d_affaires                      1575918
marge_brute                             1575750
ebe                                     1573350
ebit                                    1573365
resultat_net                             190266
taux_d_endettement                       106436
ratio_de_liquidite                        28995
ratio_de_vetuste                          29362
autonomie_financiere                      30861
poids_bfr_exploitation_sur_ca               105
couverture_des_interets                   16022
caf_sur_ca                                   30
capacite_de_remboursement                  9771
marge_ebe                                     6
resultat_courant_avant_impots_sur_ca         61
poids_bfr_exploitation_sur_ca_jours         105
rotation_des_stocks_jours                 25969
credit_clients_jours                      35717
credit_fournisseurs_jours                  6206
Name: zéros passés en valeur manquante, dtype: int64

'process : 3.20 Go | disponible : 7.08 Go'

### Observations

- Environ **1,58 million** de chiffres d'affaires, marges et EBE à 0 sont passés en valeur manquante, parce qu'ils venaient de comptes confidentiels.
- Pour le résultat net, c'est beaucoup moins (190 266), puisqu'il est souvent publié même quand le reste est caché.

### 3.4 Winsorisation (décision 4)

In [12]:
seuils = r[NUM].quantile([0.01, 0.99]).T
seuils.columns = ["limite basse (1 %)", "limite haute (99 %)"]
for col in NUM:
    bas, haut = seuils.loc[col]
    v = r[col]
    r[col + "_ramene"] = (v < bas) | (v > haut)
    r[col] = v.clip(bas, haut)
seuils["valeurs ramenées"] = [r[c + "_ramene"].sum() for c in NUM]
seuils["% ramené"] = [100 * r[c + "_ramene"].mean() for c in NUM]
seuils

,limite basse (1 %),limite haute (99 %),valeurs ramenées,% ramené
chiffre_d_affaires,0.00,"77,610,976.00",55644,0.84
marge_brute,-10.00,"42,519,060.00",100256,1.52
ebe,"-1,181,335.62","6,110,950.00",100878,1.52
ebit,"-1,324,401.00","4,126,147.25",100878,1.52
resultat_net,"-1,479,782.88","4,190,877.25",128540,1.94
taux_d_endettement,"-2,711.07","4,597.90",130142,1.97
ratio_de_liquidite,0.00,"31,913.13",68270,1.03
ratio_de_vetuste,0.00,100.00,6204,0.09
autonomie_financiere,-371.12,122.46,131684,1.99
poids_bfr_exploitation_sur_ca,"-1,276.20","2,752.25",84512,1.28


### Observations

- Selon la colonne, entre 0,1 % et 2 % des valeurs ont été ramenées. On peut toujours les retrouver avec les colonnes `<colonne>_ramene`.
- Comme prévu, la **limite basse vaut 0** pour le chiffre d'affaires, la liquidité, la vétusté, la rotation des stocks et les délais clients et fournisseurs. Les valeurs négatives impossibles sont donc ramenées à 0.
- La limite basse de la marge brute vaut −10 €. Elle peut être un peu négative, c'est possible si l'entreprise a revendu à perte.
- ⚠️ Pour le **chiffre d'affaires**, la limite haute est **77,6 M€** : les 55 644 exercices au-dessus sont ramenés à 77,6 M€. Ça corrige les montants absurdes, mais ça écrase aussi les **vraies grandes entreprises**, qui auront toutes le même chiffre d'affaires. Or c'est cette colonne qui sert à mesurer la taille. La colonne `chiffre_d_affaires_ramene` permet de les repérer.

In [13]:
r = r.reset_index(drop=True)
print("lignes :", avant, "->", len(r), "| entreprises :", r["siren"].nunique())
print(f"{r.memory_usage(deep=True).sum() / 1e9:.2f} Go |", mem())
r.describe().T[["min", "25%", "50%", "75%", "max"]]

lignes : 6813145 -> 6617202 | entreprises : 1592767
0.75 Go | process : 3.94 Go | disponible : 6.55 Go


,min,25%,50%,75%,max
siren,"5,420,120.00","441,335,619.00","529,206,277.50","820,347,490.75","999,990,542.00"
date_cloture_exercice,2016-01-01 00:00:00,2018-12-31 00:00:00,2020-12-31 00:00:00,2023-08-31 00:00:00,2026-07-31 00:00:00
chiffre_d_affaires,0.00,"29,206.75","201,154.00","1,170,468.75","77,610,976.00"
marge_brute,-10.00,"23,634.00","150,944.00","778,960.00","42,519,060.00"
ebe,"-1,181,335.62","-3,426.00","10,163.00","75,054.00","6,110,950.00"
ebit,"-1,324,401.00","-5,085.00","5,380.00","53,292.00","4,126,147.25"
resultat_net,"-1,479,782.88",-787.00,"18,477.00","94,562.00","4,190,877.25"
taux_d_endettement,"-2,711.07",0.07,18.10,88.74,"4,597.90"
ratio_de_liquidite,0.00,115.92,201.29,401.42,"31,913.13"
ratio_de_vetuste,0.00,17.26,35.98,61.10,100.00


In [14]:
r.to_parquet(PROCESSED / "ratios_clean.parquet", index=False)
print("enregistré :", (PROCESSED / "ratios_clean.parquet").stat().st_size / 1e6, "Mo")
del r
print(free_memory())

enregistré : 367.306832 Mo
process : 3.10 Go | disponible : 7.33 Go


### Observations

- Au final, on passe de **6 813 145 à 6 617 202 exercices**, pour **1 592 767 entreprises**. Le fichier `ratios_clean.parquet` fait 367 Mo.

## 4. BODACC

Les décisions (voir le récapitulatif de `bodacc.ipynb`) :

| # | Décision |
|---|---|
| 1 | Les **rectificatifs** remplacent l'annonce d'origine. |
| 2 | Les **annulations** retirent l'annonce d'origine, et ne sont pas gardées elles-mêmes. |
| 3, 16 | **Rétractations** et **arrêts d'appel infirmant** : traités comme des annulations. On retire le jugement de la même entreprise daté exactement de la date citée dans le texte. Si on ne le trouve pas, on ne retire rien. |
| 4, 12-15 | Variable **type d'événement** : signal précoce, entrée en défaillance, aggravation, issue. |
| 5 | Les **extensions** sont des entrées en défaillance pour l'entreprise visée. |
| 6 | Une entreprise peut défaillir **plusieurs fois** : on garde chaque défaillance. |
| 7 | On **garde** les annonces sans SIREN. |
| 8 | **Une ligne par SIREN**, avec un indicateur « annonce collective ». |
| 9 | Date de l'événement = **date du jugement**, sauf si elle est à plus d'**1 an** de la parution : on prend alors la date de parution. |
| 11 | On **répare l'encodage** de 2016-2018. |
| 18 | La **reprise d'une liquidation** est une **aggravation**. |
| 19 | On **corrige les dates de parution** de 2008-2016, où le jour et le mois sont inversés (correction technique). |
| 20 | Un jugement daté **après sa parution** a une date aberrante : on prend la date de parution. |
| 21 | L'**extension d'une sauvegarde** est un **signal précoce**, pas une entrée en défaillance. |
| 22 | Le faux SIREN `000000000` passe en valeur manquante. |
| 23 | On **harmonise les noms de tribunaux** au format « TYPE - VILLE ». Un tribunal qui a changé de nom est fusionné : les Tribunaux des Activités Économiques (2025) avec les Tribunaux de Commerce, les Tribunaux de Grande Instance avec les Tribunaux Judiciaires (2020). Le nom d'origine reste dans `tribunal_origine`. |

### 4.1 Lecture des fichiers, année par année

In [15]:
import json
import re
from pathlib import Path

RAW_BODACC = Path("../data/raw/bodacc")

def reparer(s):
    """Répare l'encodage cassé ('procÃ©dure' -> 'procédure') quand c'est possible."""
    if not isinstance(s, str) or "Ã" not in s:
        return s
    try:
        return s.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

def lire_annee(f):
    d = pd.read_csv(f, sep=";", dtype=str, usecols=["id", "dateparution", "typeavis_lib", "numerodepartement",
                                                    "tribunal", "registre", "listepersonnes", "jugement",
                                                    "parutionavisprecedent"])
    for c in ["typeavis_lib", "tribunal", "jugement"]:
        d[c] = d[c].map(reparer)
    j = d["jugement"].map(lambda s: json.loads(s) if isinstance(s, str) else {})
    d["famille"] = j.map(lambda x: x.get("famille"))
    d["nature"] = j.map(lambda x: x.get("nature"))
    d["date_jugement_texte"] = j.map(lambda x: x.get("date"))
    d["texte"] = j.map(lambda x: x.get("complementJugement"))
    d["type_personne"] = d["listepersonnes"].str.extract(r'"typePersonne": "(\w+)"', expand=False)
    return d.drop(columns=["jugement", "listepersonnes"])

b = pd.concat([lire_annee(f) for f in sorted(RAW_BODACC.glob("procedures_collectives_*.csv.gz"))], ignore_index=True)
print(b.shape, "|", mem())
print("textes encore cassés :", b["nature"].str.contains("Ã", na=False).sum())
b["typeavis_lib"].value_counts()

(3365280, 12) | process : 5.87 Go | disponible : 4.68 Go
textes encore cassés : 0


typeavis_lib
Avis initial         3351113
Avis rectificatif      11887
Avis d’annulation       2280
Name: count, dtype: int64

### Observations

- On lit les **3 365 280 annonces** de 2008 à 2026. Après réparation, il ne reste **plus aucun texte avec un encodage cassé** (décision 11).

### 4.2 Rectificatifs et annulations (décisions 1 et 2)

In [16]:
def id_origine(s):
    if not isinstance(s, str):
        return None
    d = json.loads(s)
    return "A" + d.get("numeroParution", "") + d.get("numeroAnnonce", "")

b["id_origine"] = b["parutionavisprecedent"].map(id_origine)
rectif = b["typeavis_lib"] == "Avis rectificatif"
annul = b["typeavis_lib"].str.contains("annulation", na=False)
visees = set(b.loc[rectif | annul, "id_origine"].dropna())
retrouvees = b["id"].isin(visees)
print("rectificatifs :", rectif.sum(), "| annulations :", annul.sum())
print("annonces d'origine retrouvées et retirées :", retrouvees.sum(), "sur", len(visees), "visées")

b = b[~retrouvees & ~annul].drop(columns=["parutionavisprecedent", "id_origine"])
print("annonces restantes :", len(b))

rectificatifs : 11887 | annulations : 2280
annonces d'origine retrouvées et retirées : 13202 sur 13711 visées


annonces restantes : 3349814


### Observations

- On a 11 887 rectificatifs et 2 280 annulations. Ils visent 13 711 annonces d'origine, et on en retrouve **13 202**. Ces annonces d'origine sont retirées : les rectificatifs les remplacent, et les annulations disparaissent avec elles.
- Les 509 annonces d'origine non retrouvées visent probablement des annonces d'avant 2008, qui ne sont pas dans nos fichiers.

### 4.3 La date de l'événement (décision 9)

In [17]:
MOIS = {"janvier": 1, "février": 2, "fevrier": 2, "mars": 3, "avril": 4, "mai": 5, "juin": 6, "juillet": 7,
        "août": 8, "aout": 8, "septembre": 9, "octobre": 10, "novembre": 11, "décembre": 12, "decembre": 12}
RE_LETTRES = re.compile(r"(\d{1,2})(?:er)?\s+([a-zéèûô]+)\s+(\d{4})")
RE_CHIFFRES = re.compile(r"(\d{1,2})\s*/+\s*(\d{1,2})\s*/+\s*(\d{4})")

def lire_date(s):
    """'2016-11-03', '7 novembre 2014', '1er mars 2022' ou '01/02/2012' -> date.
    Sinon NaT. Une année hors 1800-2100 est considérée comme illisible."""
    if not isinstance(s, str):
        return pd.NaT
    s = s.strip().lower()
    try:
        if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
            d = pd.Timestamp(s)
            return d if 1800 <= d.year <= 2100 else pd.NaT
        m = RE_LETTRES.search(s)
        if m and m.group(2) in MOIS:
            a = int(m.group(3))
            return pd.Timestamp(a, MOIS[m.group(2)], int(m.group(1))) if 1800 <= a <= 2100 else pd.NaT
        m = RE_CHIFFRES.search(s)
        if m:
            a = int(m.group(3))
            return pd.Timestamp(a, int(m.group(2)), int(m.group(1))) if 1800 <= a <= 2100 else pd.NaT
    except (ValueError, OverflowError):
        pass
    return pd.NaT

b["date_parution"] = pd.to_datetime(b["dateparution"], format="%Y-%m-%d")
b["date_jugement"] = pd.to_datetime(b["date_jugement_texte"].map(lire_date)).astype("datetime64[ns]")

# Correction technique de la source : de 2008 à 2016, le jour et le mois de la date de parution
# sont inversés quand le jour vaut 12 ou moins (ex. '2012-04-12' au lieu du 4 décembre 2012).
p = b["date_parution"]
ambigu = (p.dt.day <= 12) & (p.dt.day != p.dt.month)
inversee = pd.to_datetime(dict(year=p.dt.year, month=p.dt.day.where(ambigu, p.dt.month),
                               day=p.dt.month.where(ambigu, p.dt.day)))
plausible = lambda d: (d - b["date_jugement"]).dt.days.between(0, 365)
corriger = ambigu & ((p.dt.year <= 2015) | ((p.dt.year == 2016) & plausible(inversee) & ~plausible(p)))
b["date_parution"] = inversee.where(corriger, p)
b["parution_corrigee"] = corriger
print("dates de parution corrigées :", corriger.sum())
display(corriger.groupby(p.dt.year).sum().rename("corrigées").to_frame().T)
del p, ambigu, inversee

delai = (b["date_parution"] - b["date_jugement"]).dt.days
ok = delai.between(0, 365)
b["date_evenement"] = b["date_jugement"].where(ok, b["date_parution"])
b["date_source"] = np.select([ok, b["date_jugement"].isna()], ["jugement", "parution (date de jugement illisible)"],
                             "parution (date de jugement aberrante)")
b = b.drop(columns=["dateparution", "date_jugement_texte"])
b["date_source"].value_counts()

dates de parution corrigées : 578243


date_parution,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
corrigées,54948,70339,73486,67076,69303,73156,70785,72004,27146,0,0,0,0,0,0,0,0,0,0


date_source
jugement                                 3327706
parution (date de jugement aberrante)      20282
parution (date de jugement illisible)       1826
Name: count, dtype: int64

In [18]:
# Les dates de jugement aberrantes, en détail
print("jugement après la parution :", (delai < 0).sum())
print("jugement plus d'1 an avant la parution :", (delai > 365).sum())
# Par famille : part de dates aberrantes
(b.assign(aberrante=b["date_source"] == "parution (date de jugement aberrante)")
   .groupby("famille")["aberrante"].agg(["sum", "mean"]).round(3))

jugement après la parution : 1673
jugement plus d'1 an avant la parution : 18609


,sum,mean
famille,,
Arrêt de la Cour d'Appel,917,0.10
Avis de dépôt,13620,0.02
Extrait de jugement,2320,0.01
Jugement d'ouverture,1307,0.00
Jugement de clôture,1475,0.00
Jugement prononçant,620,0.00
Loi de 1967,17,0.02
Rétractation sur tierce opposition,6,0.01


### Observations

- ⚠️ **On a trouvé un bug dans la source** : de 2008 à 2016, le **jour et le mois de la date de parution sont inversés** quand le jour vaut 12 ou moins. Par exemple, une parution notée « 2012-04-12 » pour un jugement du 13 novembre 2012 correspond en fait au 4 décembre 2012. On l'a vu parce que ces dates donnaient des jugements publiés *avant* d'avoir eu lieu.
- Le bug est **systématique jusqu'en 2015** et s'arrête en décembre 2016. On a corrigé **578 243 dates de parution**. La colonne `parution_corrigee` permet de les retrouver.
- Une fois corrigé, la date du jugement est utilisable pour **3 327 706 annonces** (99,4 %). On prend la date de parution pour 20 282 annonces où la date du jugement est aberrante (dont 1 673 jugements datés après leur parution), et pour 1 826 où elle est illisible.
- Les dates aberrantes viennent surtout des **avis de dépôt** (13 620). Pour eux, le champ « date » contient parfois une autre date que celle d'un jugement (une date de dépôt ou une date limite).

### 4.4 Le type d'événement (décisions 4, 5, 12 à 15)

In [19]:
ENTREE, SIGNAL, AGGRAV, ISSUE, REPRISE, AUTRE = ("entrée en défaillance", "signal précoce", "aggravation",
                                                 "issue", "reprise de liquidation", "autre")

def categorie_nature(n):
    """Classe une nature de jugement précise."""
    if not isinstance(n, str):
        return AUTRE
    n = n.lower()
    if "rétractation" in n or "infirmant" in n:
        return "rétractation / infirmation"
    if "reprise de la procédure de liquidation" in n:
        return REPRISE
    if "conversion" in n or ("résolution" in n and ("et la liquidation" in n or "et le redressement" in n)):
        return AGGRAV
    if "mettant fin à la procédure de traitement de sortie de crise et ouvrant" in n:
        return AGGRAV
    if n.startswith(("jugement d'ouverture", "jugement d'extension")):
        if "sauvegarde" in n:
            return SIGNAL
        if "redressement" in n or "liquidation" in n or "sortie de crise" in n:
            return ENTREE
    if "clôture" in n or "mettant fin à la procédure" in n:
        return ISSUE
    if ("arrêtant" in n and "plan" in n) or n.startswith("jugement de plan"):
        return ISSUE
    return AUTRE

def categorie_texte(famille, texte):
    """Classe les natures 'fourre-tout' à partir du texte du jugement (décision 12)."""
    t = texte.lower() if isinstance(texte, str) else ""
    if "réouverture" in t or "reprise de la procédure" in t:
        return REPRISE
    if "conversion" in t or "résolution" in t:
        return AGGRAV
    if "sauvegarde" in t and ("ouverture" in t or "ouvrant" in t):
        return SIGNAL
    if "redressement" in t or "liquidation" in t or "sortie de crise" in t:
        nouvelle = any(m in t for m in ["ouverture", "ouvrant", "cessation des paiements", "insolvabilité"])
        if famille == "Jugement d'ouverture" or nouvelle:
            return ENTREE
        return AGGRAV
    if "plan" in t and ("adoption" in t or "arrêt" in t):
        return ISSUE
    return AUTRE

natures = pd.Series(b["nature"].dropna().unique())
table = dict(zip(natures, natures.map(categorie_nature)))
b["categorie"] = b["nature"].map(table).fillna(AUTRE)
fourre_tout = b["nature"].isin(["Autre jugement d'ouverture", "Autre jugement prononçant"])
b.loc[fourre_tout, "categorie"] = [categorie_texte(f, t) for f, t in zip(b.loc[fourre_tout, "famille"], b.loc[fourre_tout, "texte"])]
print("reprises de liquidation :", (b["categorie"] == REPRISE).sum())
b["categorie"] = b["categorie"].replace(REPRISE, AGGRAV)  # décision 18 : la reprise d'une liquidation est une aggravation
pd.crosstab(b["famille"], b["categorie"], margins=True)

reprises de liquidation : 8366


categorie,aggravation,autre,entrée en défaillance,issue,rétractation / infirmation,signal précoce,All
famille,,,,,,,
Arrêt de la Cour d'Appel,0,4556,0,0,4145,0,8701
Avis de dépôt,0,737289,0,0,0,2,737291
Extrait de jugement,48763,147663,2,115689,0,0,312117
Jugement d'ouverture,2329,1355,990978,2,0,24466,1019130
Jugement de clôture,0,15,0,997606,0,0,997621
Jugement prononçant,243597,2165,26919,152,0,24,272857
Loi de 1967,0,98,0,728,0,0,826
Rétractation sur tierce opposition,0,0,0,0,1271,0,1271
All,294689,893141,1017899,1114177,5416,24492,3349814


### Observations

- On trouve environ **1 million d'entrées en défaillance**, 295 000 aggravations (dont 8 366 reprises de liquidation), 1,1 million d'issues et 24 000 signaux précoces. Le reste (889 000) est classé « autre » : surtout les avis de dépôt, les interdictions de gérer et les modifications de plan.

In [20]:
# Le détail des natures classées dans chaque catégorie importante (hors fourre-tout)
pd.set_option("display.max_colwidth", 110)
(b[~fourre_tout & b["categorie"].isin([ENTREE, SIGNAL, AGGRAV, ISSUE, REPRISE])]
   .groupby(["categorie", "nature"]).size().rename("annonces").to_frame())

annonces
categorie             nature                                                                                                                  
aggravation           Jugement de conversion en liquidation judiciaire                                                                  210955
                      Jugement de conversion en liquidation judiciaire de la procédure de sauvegarde                                      4215
                      Jugement de conversion en liquidation judiciaire de la procédure de sauvegarde financière accélérée                    2
                      Jugement de conversion en redressement judiciaire de la procédure de sauvegarde                                     3700
                      Jugement de conversion en redressement judiciaire de la procédure de sauvegarde financière accélérée                   5
                      Jugement de reprise de la procédure de liquidation judiciaire                                                       7042
                      Jugement mettant fin à la procédure de traitement de sortie de crise et ouvrant une procédure de liquidatio...         4
                      Jugement mettant fin à la procédure de traitement de sortie de crise et ouvrant une procédure de redresseme...        19
                      Jugement prononçant la résolution du plan de cession et la liquidation judiciaire                                   1228
                      Jugement prononçant la résolution du plan de redressement et la liquidation judiciaire                             41725
                      Jugement prononçant la résolution du plan de sauvegarde accélérée et la liquidation judiciaire                        10
                      Jugement prononçant la résolution du plan de sauvegarde accélérée et le redressement judiciaire                        1
                      Jugement prononçant la résolution du plan de sauvegarde et la liquidation judiciaire                                2174
                      Jugement prononçant la résolution du plan de sauvegarde et le redressement judiciaire                               1478
                      Jugement prononçant la résolution du plan de sauvegarde financière accélérée et la liquidation judiciaire              6
                      Jugement prononçant la résolution du plan de sauvegarde financière accélérée et le redressement judiciaire             3
                      Jugement prononçant la résolution du plan de traitement de sortie de crise et la liquidation judiciaire               19
                      Jugement prononçant la résolution du plan de traitement de sortie de crise et le redressement judiciaire               5
entrée en défaillance Jugement d'extension d'une procédure de redressement judiciaire                                                     2047
                      Jugement d'extension de liquidation judiciaire                                                                      5173
                      Jugement d'ouverture d'une procédure de redressement judiciaire                                                   310987
                      Jugement d'ouverture d'une procédure de traitement de sortie de crise                                                 92
                      Jugement d'ouverture de liquidation judiciaire                                                                    656836
issue                 Autre jugement de clôture                                                                                           3304
                      Clôture de la procédure à défaut d'arrêté du plan dans le délai                                                        2
                      Jugement arrêtant le plan de sauvegarde                                                                            11929
                      Jugement arrêtant le plan de sauvegarde accélérée                                                           

In [21]:
# Les fourre-tout : comment ils ont été classés
pd.crosstab(b.loc[fourre_tout, "nature"], b.loc[fourre_tout, "categorie"])

categorie,aggravation,autre,entrée en défaillance,issue,signal précoce
nature,,,,,
Autre jugement d'ouverture,2329,225,15843,2,83
Autre jugement prononçant,19769,2137,26921,149,24


### Observations

- Grâce aux mots-clés, la plupart des « fourre-tout » sont classés. « Autre jugement d'ouverture » : 15 843 entrées en défaillance et 2 329 aggravations (dont 1 125 réouvertures de liquidation). « Autre jugement prononçant » : 26 921 entrées et 19 769 aggravations.
- Seuls 225 et 2 137 restent en « autre », parce que le texte ne contient aucun mot-clé reconnu.

### 4.5 Une ligne par entreprise (décisions 7 et 8)

In [22]:
b["sirens"] = b["registre"].str.findall(r"\b\d{9}\b").map(lambda l: sorted(set(l)) if isinstance(l, list) else [])
b["nb_siren"] = b["sirens"].str.len()
b["annonce_collective"] = b["nb_siren"] > 1
b = b.explode("sirens").rename(columns={"sirens": "siren"})
b["siren"] = pd.to_numeric(b["siren"]).astype("Int64")
print("faux SIREN 000000000 :", (b["siren"] == 0).sum())
b.loc[b["siren"] == 0, "siren"] = pd.NA  # pas un vrai SIREN
b = b.drop(columns=["registre", "nb_siren"]).reset_index(drop=True)
print("lignes :", len(b), "| sans SIREN :", b["siren"].isna().sum(), "| issues d'une annonce collective :", b["annonce_collective"].sum())

faux SIREN 000000000 : 310
lignes : 3363961 | sans SIREN : 312738 | issues d'une annonce collective : 23065


### Observations

- Une ligne par entreprise donne **3 363 961 lignes**. 312 738 n'ont pas de SIREN : elles sont gardées pour les comptages globaux. 23 065 lignes viennent d'annonces qui citent plusieurs entreprises.
- 310 annonces avaient un faux SIREN `000000000`. On l'a remplacé par une valeur manquante, sinon on aurait eu une « entreprise » avec 46 défaillances.

### 4.6 Rétractations et arrêts infirmant (décisions 3 et 16)

In [23]:
retract = b["categorie"] == "rétractation / infirmation"

def date_visee(texte):
    """La date de la décision annulée : celle qui suit 'décision' dans le texte."""
    if not isinstance(texte, str):
        return pd.NaT
    t = texte.lower()
    i = t.find("décision")
    return lire_date(t[i:]) if i >= 0 else pd.NaT

cibles = b.loc[retract & b["siren"].notna(), ["siren", "texte"]].copy()
cibles["date_visee"] = cibles["texte"].map(date_visee)
print("rétractations / infirmations :", retract.sum(), "| avec SIREN :", len(cibles), "| date visée lue :", cibles["date_visee"].notna().sum())

cles = set(zip(cibles["siren"], cibles["date_visee"]))
a_retirer = ~retract & pd.Series([(s, d) in cles for s, d in zip(b["siren"], b["date_jugement"])], index=b.index)
print("jugements retrouvés et retirés :", a_retirer.sum())
display(b.loc[a_retirer, "categorie"].value_counts())
b = b[~a_retirer & ~retract]

rétractations / infirmations : 5517 | avec SIREN : 4946 | date visée lue : 4300


jugements retrouvés et retirés : 3683


categorie
entrée en défaillance    2031
autre                     806
aggravation               748
issue                      82
signal précoce             16
Name: count, dtype: int64

### Observations

- On a 5 517 rétractations et arrêts infirmant. Pour 4 300 d'entre eux, on arrive à lire la date de la décision annulée dans le texte, et on retrouve le jugement visé dans **3 683 cas**. Il s'agit surtout d'entrées en défaillance (2 031), ce qui est logique : une rétractation annule souvent une ouverture de procédure.
- Les rétractations et arrêts eux-mêmes ne sont pas gardés.

### 4.7 Doublons, tribunaux et enregistrement

In [24]:
import unicodedata

# Décision BODACC 23 : harmoniser les noms de tribunaux -> 'TYPE - VILLE'
TYPES_TRIBUNAL = [  # (début du nom normalisé, type harmonisé)
    ("TRIBUNAL DES ACTIVITES ECONOMIQUES", "TRIBUNAL DE COMMERCE"),   # nouveau nom des tribunaux de commerce depuis 2025
    ("TRIBUNAL DE COMMERCE", "TRIBUNAL DE COMMERCE"),
    ("TRIBUNAL MIXTE DE COMMERCE", "TRIBUNAL MIXTE DE COMMERCE"),
    ("TRIBUNAL DE GRANDE INSTANCE", "TRIBUNAL JUDICIAIRE"),           # devenus tribunaux judiciaires en 2020
    ("TRIBUNAL JUDICIAIRE", "TRIBUNAL JUDICIAIRE"),
    ("TRIBUNAL D INSTANCE", "TRIBUNAL D INSTANCE"),
    ("TRIBUNAL DE PREMIERE INSTANCE", "TRIBUNAL DE PREMIERE INSTANCE"),
    ("TRIBUNAL DE PROXIMITE", "TRIBUNAL DE PROXIMITE"),
    ("CHAMBRE COMMERCIALE", "CHAMBRE COMMERCIALE"),
    ("COUR D APPEL", "COUR D APPEL"),
    ("TRIBUNAL SUPERIEUR D APPEL", "TRIBUNAL SUPERIEUR D APPEL"),
]

def harmoniser_tribunal(nom):
    """'Greffe du Tribunal des Activités Economiques de Paris' -> 'TRIBUNAL DE COMMERCE - PARIS'."""
    if not isinstance(nom, str):
        return nom
    s = unicodedata.normalize("NFKD", nom).encode("ascii", "ignore").decode().upper()
    s = re.sub(r"[’'`.]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"^\d+", "", s).strip()
    s = re.sub(r"^GREFFE (DU|DE LA|DES|DE L|DE)\s+", "", s)
    s = re.sub(r"\bST-", "SAINT-", s)
    for debut, type_ in TYPES_TRIBUNAL:
        if s.startswith(debut):
            ville = re.sub(r"^(DE LA|DE L|DES|DU|DE|D)\s+", "", s[len(debut):].strip())
            ville = re.sub(r"\s+\d+$", "", ville).strip()   # 'PARIS 02' -> 'PARIS'
            return f"{type_} - {ville}" if ville else type_
    return s

b["tribunal_origine"] = b["tribunal"]
noms = pd.Series(b["tribunal"].dropna().unique())
b["tribunal"] = b["tribunal"].map(dict(zip(noms, noms.map(harmoniser_tribunal))))
print("tribunaux : ", b["tribunal_origine"].nunique(), "noms ->", b["tribunal"].nunique(), "tribunaux harmonisés")
display(b["tribunal"].str.split(" - ").str[0].value_counts().rename("annonces").to_frame())

# Le même jugement publié deux fois pour la même entreprise (même nature, même date)
cle = ["siren", "nature", "date_jugement"]
doublon = b["siren"].notna() & b["date_jugement"].notna() & b.duplicated(cle, keep="first")
print("doublons retirés :", doublon.sum())
display(b.loc[doublon, "categorie"].value_counts())
b = b[~doublon].drop(columns="texte").reset_index(drop=True)

for c in ["typeavis_lib", "numerodepartement", "tribunal", "tribunal_origine", "famille", "nature", "type_personne", "categorie", "date_source"]:
    b[c] = b[c].astype("category")
print(b.shape, f"{b.memory_usage(deep=True).sum() / 1e9:.2f} Go |", mem())
b["categorie"].value_counts()

tribunaux :  1056 noms -> 477 tribunaux harmonisés


,annonces
tribunal,
TRIBUNAL DE COMMERCE,2856371
TRIBUNAL JUDICIAIRE,432679
TRIBUNAL MIXTE DE COMMERCE,63589
CHAMBRE COMMERCIALE,1440
TRIBUNAL D INSTANCE,392
TRIBUNAL DE PREMIERE INSTANCE,227
COUR D APPEL,54
TRIBUNAL DE PROXIMITE,6
TRIBUNAL SUPERIEUR D APPEL,3


doublons retirés : 14708


categorie
autre                    9557
issue                    2237
entrée en défaillance    2063
aggravation               718
signal précoce            133
Name: count, dtype: int64

(3340053, 16) 0.22 Go | process : 6.62 Go | disponible : 4.06 Go


categorie
issue                    1115676
entrée en défaillance    1016272
autre                     889094
aggravation               294549
signal précoce             24462
Name: count, dtype: int64

### Observations

- 14 708 annonces étaient publiées **deux fois** pour le même jugement de la même entreprise. On n'en garde qu'une.
- Les noms de tribunaux passent de **1 056 variantes à 477 tribunaux**. Par exemple, « TRIBUNAL DE COMMERCE DE PARIS », « Greffe du Tribunal de Commerce de Paris » et « Greffe du Tribunal des Activités Economiques de Paris » deviennent tous « TRIBUNAL DE COMMERCE - PARIS ». La grande majorité des annonces vient des tribunaux de commerce (2,86 M), puis des tribunaux judiciaires (433 000, pour les entreprises non commerçantes).
- Le fichier `bodacc_clean.parquet` contient **3 340 053 lignes** (59 Mo).

In [25]:
# Entrées en défaillance par année (avec SIREN ou non)
entrees = b[b["categorie"] == ENTREE]
pd.crosstab(entrees["date_evenement"].dt.year, entrees["siren"].isna().map({False: "avec SIREN", True: "sans SIREN"}), margins=True)

siren,avec SIREN,sans SIREN,All
date_evenement,,,
2007,1452,358,1810
2008,48621,7063,55684
2009,55176,7144,62320
2010,52685,6276,58961
2011,52271,5942,58213
2012,53422,5691,59113
2013,55037,5555,60592
2014,53332,5493,58825
2015,53735,5706,59441


In [26]:
# Défaillances multiples (décision 6) : combien d'entrées en défaillance par entreprise ?
entrees.dropna(subset=["siren"]).groupby("siren").size().value_counts().sort_index()

1    899051
2     14950
3       371
4        32
5         4
6         1
Name: count, dtype: int64

### Observations

- Les entrées en défaillance sont stables autour de **50 000 à 60 000 par an** de 2008 à 2019. Elles chutent pendant le **Covid** (30 373 en 2020, 26 795 en 2021), puis remontent fortement : **65 220 en 2025**, le plus haut niveau de la période. 2026 s'arrête fin octobre (50 585).
- La grande majorité des entreprises n'a **qu'une seule** entrée en défaillance (899 051). 14 950 en ont deux, et au maximum une entreprise en a six.

In [27]:
b.to_parquet(PROCESSED / "bodacc_clean.parquet", index=False)
print("enregistré :", (PROCESSED / "bodacc_clean.parquet").stat().st_size / 1e6, "Mo")
entrees = entrees.loc[entrees["siren"].notna(), ["siren", "date_evenement"]].copy()
entrees["siren"] = entrees["siren"].astype("int64")
del b
print(free_memory())

enregistré : 62.607038 Mo
process : 3.28 Go | disponible : 7.38 Go


## 5. Assemblage : une ligne par exercice comptable

Les décisions (voir `ratios.ipynb` et `sirene.ipynb`) :

| Décision | Ce qu'on fait |
|---|---|
| Ratios 6 | **Une ligne = un exercice**. Étiquette `faillite_36_mois` : y a-t-il une **entrée en défaillance** dans les 36 mois **après** la date de clôture ? Les exercices clôturés **après fin octobre 2023** sont mis de côté, car leurs 36 mois ne sont pas finis. |
| BODACC 17 | Seules les **entrées en défaillance** comptent pour l'étiquette. |
| SIRENE 6 | **Âge de l'entreprise** à la date de clôture des comptes. |
| Ratios 8 | Taille = chiffre d'affaires **après winsorisation**. |
| Ratios 9 | On **garde** les exercices d'entreprises déjà passées par une défaillance (colonne `deja_defaillante`). |
| SIRENE 9 | Un **âge négatif** (création SIRENE après la clôture) passe en valeur manquante. |

In [28]:
r = pd.read_parquet(PROCESSED / "ratios_clean.parquet")
limite = DATE_FIN_DONNEES - pd.DateOffset(months=36)
trop_recent = r["date_cloture_exercice"] > limite
print(f"exercices clôturés après le {limite.date()} mis de côté :", trop_recent.sum(), "sur", len(r))
r = r[~trop_recent]
r["date_cloture_exercice"] = r["date_cloture_exercice"].astype("datetime64[ns]")
entrees["date_evenement"] = entrees["date_evenement"].astype("datetime64[ns]")

exercices clôturés après le 2023-10-31 mis de côté : 1559227 sur 6617202


In [29]:
# Prochaine entrée en défaillance après chaque clôture (même entreprise)
r = r.sort_values("date_cloture_exercice")
e = entrees.sort_values("date_evenement").rename(columns={"date_evenement": "date_prochaine_defaillance"})
r = pd.merge_asof(r, e, left_on="date_cloture_exercice", right_on="date_prochaine_defaillance",
                  by="siren", direction="forward", allow_exact_matches=False)
mois = (r["date_prochaine_defaillance"] - r["date_cloture_exercice"]).dt.days / 30.44
r["mois_avant_defaillance"] = mois.astype("float32")
r["faillite_36_mois"] = mois <= 36

# Une entrée en défaillance avant la clôture : l'entreprise était déjà passée par une procédure
e2 = entrees.sort_values("date_evenement").rename(columns={"date_evenement": "date_derniere_defaillance"})
r = pd.merge_asof(r, e2, left_on="date_cloture_exercice", right_on="date_derniere_defaillance",
                  by="siren", direction="backward", allow_exact_matches=True)
r["deja_defaillante"] = r["date_derniere_defaillance"].notna()
del e, e2, entrees
free_memory()
r["faillite_36_mois"].value_counts()

faillite_36_mois
False    4961766
True       96209
Name: count, dtype: int64

### Observations

- Sur 5 057 975 exercices clôturés avant novembre 2023, **96 209 (1,9 %)** sont suivis d'une entrée en défaillance dans les 36 mois.
- Les 1 559 227 exercices clôturés après fin octobre 2023 sont mis de côté, parce que leurs 36 mois ne sont pas encore finis.

In [30]:
# On ajoute les infos SIRENE
s = pd.read_parquet(PROCESSED / "sirene_clean.parquet",
                    columns=["siren", "date_creation", "categorie_juridique", "cj_famille", "naf", "naf_division",
                             "naf_section", "departement", "active"])
s = s[s["siren"].isin(r["siren"].unique())]
r = r.merge(s, on="siren", how="left")
del s
r["age_ans"] = ((r["date_cloture_exercice"] - r["date_creation"]).dt.days / 365.25).astype("float32")
r = r.sort_values(["siren", "date_cloture_exercice"]).reset_index(drop=True)
print(r.shape, f"{r.memory_usage(deep=True).sum() / 1e9:.2f} Go |", mem())
print("entreprises absentes de SIRENE :", r["cj_famille"].isna().sum(), "exercices")
negatif = r["age_ans"] < 0
print("âge négatif (création après la clôture), passé en valeur manquante :", negatif.sum())
r.loc[negatif, "age_ans"] = np.nan  # décision SIRENE 9
r[["age_ans", "mois_avant_defaillance"]].describe()

(5057975, 56) 0.97 Go | process : 6.39 Go | disponible : 4.10 Go
entreprises absentes de SIRENE : 83 exercices
âge négatif (création après la clôture), passé en valeur manquante : 727


,age_ans,mois_avant_defaillance
count,"5,057,144.00","285,383.00"
mean,14.59,50.83
std,14.46,26.77
min,0.00,0.03
25%,4.54,29.37
50%,10.41,48.16
75%,20.04,70.24
max,123.83,127.27


### Observations

- Seulement 83 exercices n'ont pas d'entreprise correspondante dans SIRENE.
- **727 exercices ont un âge négatif** : la date de création dans SIRENE est après la date de clôture des comptes. C'est incohérent : ces entreprises ont des comptes jusqu'à 9 ans avant leur « création ». La date de création SIRENE correspond sans doute à un autre événement. Leur âge passe en valeur manquante.
- L'âge médian d'une entreprise à la clôture de ses comptes est de **10,4 ans**.

In [31]:
# Combien d'exercices et d'entreprises en faillite, par année de clôture
annee = r["date_cloture_exercice"].dt.year
pd.DataFrame({
    "exercices": annee.value_counts().sort_index(),
    "faillite dans les 36 mois": r.loc[r["faillite_36_mois"], "date_cloture_exercice"].dt.year.value_counts().sort_index(),
}).assign(**{"% faillite": lambda t: 100 * t["faillite dans les 36 mois"] / t["exercices"]})

,exercices,faillite dans les 36 mois,% faillite
date_cloture_exercice,,,
2016,669811,15643,2.34
2017,719754,13099,1.82
2018,698664,9510,1.36
2019,700768,9169,1.31
2020,690253,12424,1.80
2021,683861,15301,2.24
2022,676180,16119,2.38
2023,218684,4944,2.26


### Observations

- La part d'exercices suivis d'une faillite varie entre **1,3 % et 2,4 %** selon l'année de clôture. Elle est au plus bas pour les comptes de 2018-2019 (leurs 36 mois couvrent 2020-2022, les années où il y a eu le moins d'entrées en défaillance), et au plus haut pour 2016 et 2022.

In [32]:
print("exercices :", len(r), "| entreprises :", r["siren"].nunique())
print("entreprises avec au moins un exercice 'faillite' :", r.loc[r["faillite_36_mois"], "siren"].nunique())
print("exercices clôturés alors que l'entreprise était déjà passée par une défaillance :", r["deja_defaillante"].sum())
r.to_parquet(PROCESSED / "panel_exercices.parquet", index=False)
print("enregistré :", (PROCESSED / "panel_exercices.parquet").stat().st_size / 1e6, "Mo")
del r
print(free_memory())

exercices : 5057975 | entreprises : 1407147
entreprises avec au moins un exercice 'faillite' : 65466
exercices clôturés alors que l'entreprise était déjà passée par une défaillance : 39618


enregistré : 316.461621 Mo
process : 3.11 Go | disponible : 7.40 Go


### Observations

- Le fichier final `panel_exercices.parquet` contient **5 057 975 exercices** pour **1 407 147 entreprises**, dont **65 466** ont au moins un exercice suivi d'une faillite (316 Mo).
- **39 618 exercices** ont été clôturés alors que l'entreprise était **déjà passée par une défaillance** : par exemple une entreprise en plan de redressement qui continue à déposer ses comptes. La colonne `deja_defaillante` permet de les repérer.